# 5. One-stage detector without anchors: YOLO11
## How it works
YOLO has **no anchors**. The image is a grid of points. A point near the centre of an object says: "I am the centre of a car. Its edges are this far: left, right, up and down."

To choose which points learn an object, YOLO11 uses **task-aligned assignment**: points with a good class score and a good box become positives.

**Small and dense objects:** two small objects next to each other can fall on the same point. Then one object is lost.

## Experiments
* **baseline**: default settings
* **box loss x1.5**: the box error counts more in the loss (loss change)
* **SGD, lr 0.005**: other optimizer and learning rate

## Imports and paths
Libraries and the folders for the data, the training files and the figures.

In [1]:
import os
import io
import json
import time
import random
import contextlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw
import torch

In [2]:
if os.path.exists("/kaggle/input"):
    import glob
    DATA = os.path.dirname(glob.glob("/kaggle/input/**/test_images.csv", recursive=True)[0])
    WORK = "/kaggle/working/work"
elif os.name == "nt":
    DATA = "D:/hw2/data"
    WORK = "D:/hw2/work"
else:
    DATA = "/content/data"
    WORK = "/content/work"

# Figures for the report go to the images folder of the repository.
IMAGES = "../images"
RESULTS_FILE = "experiments.csv"
os.makedirs(WORK, exist_ok=True)
os.makedirs(IMAGES, exist_ok=True)

CLASSES = ["person", "bicycle", "car", "van", "truck", "tricycle", "bus", "motor"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 36
print("data:", DATA)
print("device:", DEVICE)

data: D:/hw2/data
device: cuda


## Functions
This notebook uses the functions below. Every function does one simple job.

`free_memory` cleans the memory of the GPU and the computer after a model is done.

In [3]:
def free_memory():
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

`train_ultralytics` trains YOLO or RT-DETR with Ultralytics. It returns the path of the best weights, the file with the per-epoch scores and the minutes.

In [4]:
def train_ultralytics(weights, data_yaml, size, epochs, batch, name, val_each_epoch, **params):
    """Train YOLO or RT-DETR. Returns the path of the best weights, the training curve file and minutes."""
    from ultralytics import YOLO, RTDETR

    if "rtdetr" in weights:
        model = RTDETR(weights)
    else:
        model = YOLO(weights)
    # Folder names: only letters and numbers.
    name = "".join(c if c.isalnum() else "_" for c in name)
    # On Windows every data worker takes about 1 GB of RAM.
    workers = 1 if os.name == "nt" else 4
    start = time.time()
    model.train(data=data_yaml, imgsz=size, epochs=epochs, batch=batch, seed=SEED, workers=workers,
                project=f"{WORK}/runs", name=name, exist_ok=True, plots=False, verbose=False,
                val=val_each_epoch, **params)
    minutes = (time.time() - start) / 60
    best = str(model.trainer.best)
    curve = f"{model.trainer.save_dir}/results.csv"
    del model
    free_memory()
    return best, curve, minutes

`ultralytics_predictor` turns Ultralytics weights into a function: image -> boxes, scores, classes.

In [5]:
def ultralytics_predictor(weights, size):
    """Returns a function: image -> boxes, scores, classes (0..7)."""
    from ultralytics import YOLO, RTDETR

    if "rtdetr" in str(weights):
        model = RTDETR(weights)
    else:
        model = YOLO(weights)

    def predict(image):
        # Ultralytics wants BGR colour order.
        bgr = np.ascontiguousarray(image[:, :, ::-1])
        result = model.predict(bgr, imgsz=size, conf=0.001, max_det=500, verbose=False)[0].boxes
        return result.xyxy.cpu().numpy(), result.conf.cpu().numpy(), result.cls.cpu().numpy().astype(int)

    return predict

`predict_frame` predicts one frame and keeps the 500 best boxes.

In [6]:
def predict_frame(predict, path):
    """Predict one frame. Keep the 500 boxes with the best scores (Kaggle uses at most 500)."""
    image = np.array(Image.open(path).convert("RGB"))
    boxes, scores, classes = predict(image)
    order = np.argsort(-scores)[:500]
    return boxes[order], scores[order], classes[order]

`evaluate` scores a model on the validation frames with the COCO metric (pycocotools). Ignored regions have `iscrowd = 1`, so boxes there are not errors. It also measures the speed (ms per image).

In [7]:
def evaluate(predict, val_frames):
    """
    COCO score on the validation frames.
    Ignored regions are marked iscrowd=1, so boxes inside them are not counted as errors.
    """
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval

    detections = []
    predictions = []
    start = time.time()
    for row in val_frames.itertuples():
        boxes, scores, classes = predict_frame(predict, row.path)
        predictions.append((boxes, scores, classes))
        for box, score, cls in zip(boxes, scores, classes):
            x1, y1, x2, y2 = [float(v) for v in box]
            detections.append({"image_id": int(row.image_id), "category_id": int(cls) + 1,
                               "bbox": [x1, y1, x2 - x1, y2 - y1], "score": float(score)})
    ms_per_image = (time.time() - start) / len(val_frames) * 1000

    with open(f"{DATA}/val/annotations_coco.json") as f:
        ground_truth = json.load(f)
    for a in ground_truth["annotations"]:
        a.setdefault("iscrowd", 0)
        a.setdefault("area", a["bbox"][2] * a["bbox"][3])

    result = {"mAP": 0.0, "AP50": 0.0, "AP_small": 0.0, "AP_medium": 0.0, "AP_large": 0.0}
    if len(detections) > 0:
        # pycocotools prints a lot of text, we hide it.
        with contextlib.redirect_stdout(io.StringIO()):
            gt = COCO()
            gt.dataset = ground_truth
            gt.createIndex()
            E = COCOeval(gt, gt.loadRes(detections), "bbox")
            E.params.imgIds = list(val_frames["image_id"])
            # Up to 500 boxes per frame (scenes are dense).
            E.params.maxDets = [1, 100, 500]
            E.evaluate()
            E.accumulate()
            E.summarize()
        result = {"mAP": E.stats[0], "AP50": E.stats[1], "AP_small": E.stats[3],
                  "AP_medium": E.stats[4], "AP_large": E.stats[5]}
    for key in result:
        result[key] = round(float(result[key]), 4)
    result["ms/img"] = round(ms_per_image, 1)
    return result, predictions

`save_result` adds one line to `experiments.csv`. All notebooks write to this file. We use it in the report.

In [8]:
def save_result(notebook, model, experiment, metrics, minutes):
    """Add one line to experiments.csv. All notebooks write to the same file."""
    row = {"notebook": notebook, "model": model, "experiment": experiment}
    row.update(metrics)
    row["train_min"] = round(minutes, 1)
    if os.path.exists(RESULTS_FILE):
        table = pd.read_csv(RESULTS_FILE, dtype={"notebook": str})
        table = pd.concat([table, pd.DataFrame([row])], ignore_index=True)
    else:
        table = pd.DataFrame([row])
    table.to_csv(RESULTS_FILE, index=False)

## Settings
All models in notebooks 03-06 use the same settings. So we can compare them in a fair way.

The settings are small. One experiment takes only a few minutes.

In [9]:
SIZE = 640
EPOCHS = 3
TRAIN_FOLDER = f"{WORK}/fast/train"
DATA_YAML = f"{WORK}/fast/data.yaml"
val_frames = pd.read_csv(f"{WORK}/val_frames.csv")

print("training images:", len(os.listdir(f"{TRAIN_FOLDER}/images")))
print("validation frames:", len(val_frames))

training images: 271
validation frames: 146


## Train and score
Ultralytics trains the model. We score it with our own function, the same as for the other models.

For the baseline we also save the score after every epoch. We need this curve in notebook 06.

In [10]:
import shutil

experiments = {
    "baseline": {},
    "box loss x1.5": {"box": 11.25},
    "SGD, lr 0.005": {"optimizer": "SGD", "lr0": 0.005},
}

results = []
for name in experiments:
    print("=====", name)
    val_each_epoch = name == "baseline"
    best, curve, minutes = train_ultralytics("yolo11s.pt", DATA_YAML, SIZE, EPOCHS, batch=8,
                                             name=f"yolo {name}", val_each_epoch=val_each_epoch, **experiments[name])
    if val_each_epoch:
        shutil.copy(curve, "curve_yolo.csv")
    metrics, _ = evaluate(ultralytics_predictor(best, SIZE), val_frames)
    save_result("05", "YOLO11-s", name, metrics, minutes)
    results.append({"experiment": name, **metrics, "train_min": round(minutes, 1)})
    free_memory()

pd.DataFrame(results)

===== baseline
New https://pypi.org/project/ultralytics/8.4.214 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.205  Python-3.12.5 torch-2.14.1+cu126 CUDA:0 (NVIDIA GeForce GTX 1660 Ti with Max-Q Design, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:/hw2/work/fast/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=3, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.

,experiment,mAP,AP50,AP_small,AP_medium,AP_large,ms/img,train_min
0,baseline,0.0896,0.1820,0.0277,0.0796,0.2216,171.6,2.0
1,box loss x1.5,0.0814,0.1657,0.0288,0.0695,0.2084,124.6,1.1
2,"SGD, lr 0.005",0.1159,0.2182,0.0252,0.0999,0.2605,123.6,1.0


## Compare by object size
All models from notebooks 03, 04 and 05. Look at AP_small, AP_medium and AP_large.

In [11]:
table = pd.read_csv(RESULTS_FILE, dtype={"notebook": str})
table = table[table["notebook"].isin(["03", "04", "05"])]
table[["model", "experiment", "mAP", "AP_small", "AP_medium", "AP_large", "ms/img"]].sort_values("mAP", ascending=False)

,model,experiment,mAP,AP_small,AP_medium,AP_large,ms/img
8,YOLO11-s,"SGD, lr 0.005",0.1177,0.0305,0.0855,0.2661,150.0
6,YOLO11-s,baseline,0.1164,0.0344,0.0757,0.2729,155.9
19,YOLO11-s,"SGD, lr 0.005",0.1159,0.0252,0.0999,0.2605,123.6
7,YOLO11-s,box loss x1.5,0.1084,0.0353,0.0740,0.2538,145.2
2,Faster R-CNN,small anchors + more proposals,0.0977,0.0388,0.0867,0.2325,321.2
1,Faster R-CNN,small anchors,0.0961,0.0416,0.0839,0.2243,268.0
11,Faster R-CNN,baseline,0.0953,0.0417,0.0754,0.2266,212.8
12,Faster R-CNN,small anchors,0.0950,0.0419,0.0809,0.2170,218.1
13,Faster R-CNN,small anchors + more proposals,0.0915,0.0437,0.0789,0.2195,297.9
17,YOLO11-s,baseline,0.0896,0.0277,0.0796,0.2216,171.6


## Conclusions

| experiment | mAP | AP_small | AP_medium | AP_large | ms/img |
|---|---|---|---|---|---|
| baseline | 0.0896 | 0.0277 | 0.0796 | 0.2216 | 171.6 |
| box loss x1.5 | 0.0814 | **0.0288** | 0.0695 | 0.2084 | 124.6 |
| SGD, lr 0.005 | **0.1159** | 0.0252 | **0.0999** | **0.2605** | **123.6** |
| *best Faster R-CNN (baseline)* | *0.0953* | ***0.0417*** | *0.0754* | *0.2266* | *212.8* |
| *best RetinaNet (baseline)* | *0.0383* | *0.0120* | *0.0264* | *0.1079* | *168.6* |

**Search range and selected settings**
* Box loss weight: 7.5 (default) and 11.25 (x1.5). A bigger weight did not help: mAP went down in both runs.
* Optimizer: auto (Ultralytics picks AdamW for short training) and SGD with lr 0.005. SGD was the best in both runs (0.1177 and 0.1159).
* **Selected: SGD, lr 0.005, box weight 7.5.**
* The YOLO baseline gave 0.1164 in the first run and 0.0896 in the second run. With 3 epochs the noise is big, so only differences bigger than about 0.03 are sure.

**Comparison by object size**
* YOLO has the **best mAP** and it is the **fastest** model (124 ms vs 213 ms for Faster R-CNN).
* YOLO is better on **medium** and **large** objects.
* Faster R-CNN is clearly better on **small** objects (0.042 vs 0.025-0.029).

**How YOLO assigns points to objects (no anchors)**
* The image is a grid of points on 3 levels: one point every 8, 16 and 32 px.
* For every object YOLO looks at the points inside its box. It gives every point a score: how good the class score is and how well the predicted box fits (task-aligned assignment). The best 10 points become positives.
* **Small objects:** an object of 10-15 px covers only 1-2 points even on the finest level (8 px). So it gets very few positive points. Faster R-CNN has a finer level (one point every 4 px), and this is why it is better on small objects.
* **Dense objects:** when two objects are close, they want the same points. A point can belong to only one object, so one of the objects gets fewer points or none. This makes misses in dense scenes.